# 07 - Geographic variation after controlling for subtype

Subtype composition can confound geographic comparisons because mutation pathways and sampling frames differ between subtypes. This notebook calculates direct subtype-standardized estimates and compares them with crude country estimates.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA = Path('data')
candidates = [DATA / 'hivdb_patient_resistance.csv', DATA / 'current' / 'hivdb_patient_resistance_current.csv']
path = next((p for p in candidates if p.exists()), None)
if path is None:
    print('No patient-level subtype/resistance file found.')
    print('The historical study table includes aggregate subtype distributions, which are insufficient for individual-level standardization.')
    frame = pd.DataFrame()
else:
    frame = pd.read_csv(path)
    required = {'patient_id', 'iso3', 'subtype'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if 'resistant' in frame.columns:
        frame['resistance_pct'] = pd.to_numeric(frame['resistant'], errors='coerce') * 100
    elif 'resistance_pct' not in frame.columns:
        raise ValueError('Provide a binary resistant column or compatibility resistance_pct column.')
    frame['resistance_pct'] = pd.to_numeric(frame['resistance_pct'], errors='coerce')

## Choose a reference subtype distribution

The reference distribution should be external or pre-specified, not chosen after seeing country results. Below, the default is the pooled distribution among complete records. Replace it with a surveillance target population when available.

In [ ]:
if not frame.empty:
    complete = frame.dropna(subset=['iso3', 'subtype', 'resistance_pct']).copy()
    reference = complete['subtype'].value_counts(normalize=True)
    by_subtype = (complete.groupby(['iso3', 'subtype'])
                  .agg(rate=('resistance_pct', 'mean'), n=('resistance_pct', 'size'))
                  .reset_index())
    crude = complete.groupby('iso3')['resistance_pct'].mean().rename('crude_rate')
    standardized = (by_subtype.assign(weight=by_subtype['subtype'].map(reference))
                   .dropna(subset=['weight']).assign(product=lambda d: d.rate * d.weight)
                   .groupby('iso3').agg(standardized_rate=('product', 'sum'),
                                         subtype_cells=('subtype', 'nunique'))
                   .join(crude).reset_index())
    # Do not call a country standardized if it lacks a reference subtype cell.
    standardized = standardized[standardized['subtype_cells'].eq(len(reference))].copy()
    standardized['difference_percentage_points'] = standardized['standardized_rate'] - standardized['crude_rate']
    display(standardized.sort_values('standardized_rate', ascending=False).head(30))
    fig = px.scatter(standardized, x='crude_rate', y='standardized_rate', hover_name='iso3',
                     labels={'crude_rate': 'Crude resistance (%)', 'standardized_rate': 'Subtype-standardized resistance (%)'},
                     title='Crude versus subtype-standardized country estimates')
    fig.show()
else:
    print('Standardized geographic estimates are deferred until subtype-specific person-level denominators are available.')

## Interpretation guardrails

A persistent difference after standardization is compatible with geographic variation, but it can still reflect calendar time, study site, recruitment, sequencing success, HIV-1 group or recombinant form, and unmeasured treatment history. Report confidence intervals and sensitivity analyses before calling a pattern a cluster.